# BuildArena Supported Block Preview


In [ ]:
from __future__ import annotations

from collections import Counter, defaultdict
from pathlib import Path

import tomllib

from buildarena.paths import get_block_registry_path, get_saved_machine_dir

GENERATED_REGISTRY = get_block_registry_path()
SAVED_MACHINE_DIR = get_saved_machine_dir()

from buildarena.build import Machine
from buildarena.definitions_loader import load_runtime_blocks, BLOCK_NAME_BY_ID

CATEGORY_ORDER = [
    "basic", "locomotion", "mechanical", "weaponry",
    "armour", "flight", "automation", "water", 
    "space-flight"
]

EXPECTED_CATEGORY_COUNTS = {
    "basic": 9,
    "locomotion": 12,
    "mechanical": 12,
    "weaponry": 16,
    "armour": 11,
    "flight": 9,
    "automation": 10,
    "water": 8,
    "space-flight": 12,
}

# Exclude by block ID (tight, no string matching)
EXCLUDE_IDS_FROM_RENDER: set[int] = {
    57,   # Pin
    58,   # Camera Block
}

with open(GENERATED_REGISTRY, "rb") as file:
    registry_obj = tomllib.load(file)

runtime_defs = registry_obj.get("runtime_defs", {})
if not isinstance(runtime_defs, dict):
    raise ValueError("Invalid generated registry: missing [runtime_defs]")

runtime_blocks = load_runtime_blocks(registry_path=GENERATED_REGISTRY)
runtime_blocks_by_id: dict[int, dict] = {}
for name, bdef in runtime_blocks.items():
    runtime_blocks_by_id[bdef["id"]] = bdef

render_blocks_by_category: dict[str, list[dict]] = defaultdict(list)
all_counts: Counter = Counter()

for raw_key, block_def in runtime_defs.items():
    if not isinstance(block_def, dict):
        continue
    if not bool(block_def.get("enabled", True)):
        continue

    block_id = int(raw_key) if str(raw_key).isdigit() else None

    cat_value = block_def.get("category", [])
    if isinstance(cat_value, str):
        categories = [cat_value.lower()]
    else:
        categories = [c.lower() for c in cat_value]

    rt_def = runtime_blocks_by_id.get(block_id) if block_id is not None else None
    block_type = str(rt_def.get("type", "")) if rt_def else str(block_def.get("type", ""))

    for cat_key in categories:
        all_counts[cat_key] += 1

    for cat_key in categories:
        if cat_key not in EXPECTED_CATEGORY_COUNTS:
            continue
        if block_type == "connection":
            continue
        if block_id is not None and block_id in EXCLUDE_IDS_FROM_RENDER:
            continue

        display_name = BLOCK_NAME_BY_ID.get(block_id, str(raw_key)) if block_id is not None else str(raw_key)

        if block_id is not None and block_id not in runtime_blocks_by_id:
            print(f"[not loadable] id={block_id} ({display_name})")
            continue

        render_blocks_by_category[cat_key].append({
            "block_id": block_id,
            "display_name": display_name,
        })

print("=== Strict category counts (from generated registry) ===")
strict_ok = True
for category_key in CATEGORY_ORDER:
    expected = EXPECTED_CATEGORY_COUNTS[category_key]
    actual = int(all_counts.get(category_key, 0))
    ok = expected == actual
    strict_ok = strict_ok and ok
    status = "OK" if ok else "MISMATCH"
    print(f"{status:8s} {category_key:11s} expected={expected:2d} actual={actual:2d}")

print(f"strict_check_passed={strict_ok}")

print("\n=== Renderable counts (after filtering connection + exclusions) ===")
for category_key in CATEGORY_ORDER:
    entries = render_blocks_by_category.get(category_key, [])
    names = [e["display_name"] for e in entries]
    print(f"{category_key:11s} count={len(names):2d} -> {names}")

current_height = 0.0


In [ ]:
machine = Machine(name=f"preview_all", do_collision=False, registry_path=GENERATED_REGISTRY)
machine.started = True


def build_machine_for_category(*, category_key: str, spacing: float = 5.0) -> Machine:

    x_offset = 0.0
    for entry in render_blocks_by_category.get(category_key, []):
        display_name = str(entry["display_name"])
        # print(f"Trying to add: {display_name}")
        local_id = str(machine.uid)

        # Get a free base block
        base_block = machine.blocks_storage.get(
            block_name="Starting Block",
            local_id=local_id, 
            note="base block"
        )
        base_block.shift(shift_real=[x_offset, 0.0, current_height])

        # Add base wooden block
        collision_msg = machine._add_block(block=base_block, return_summary=True)

        if display_name != "Starting Block":
            # Attach target to the base
            msg = machine.attach_block_to(
                base_block=local_id, 
                face="f4", 
                new_block=display_name)

            msg = machine.attach_block_to(
                base_block=local_id, 
                face="f5", 
                new_block=display_name)
        
            print(f"{display_name}: {msg}")

        x_offset += spacing
    
    machine.to_file(output_dir=SAVED_MACHINE_DIR)
    return machine


In [ ]:
machine = build_machine_for_category(category_key="basic")
current_height += 5.0

In [ ]:
machine = build_machine_for_category(category_key="locomotion")
current_height += 5.0

In [ ]:
machine = build_machine_for_category(category_key="mechanical")
current_height += 5.0

In [ ]:
machine = build_machine_for_category(category_key="weaponry")
current_height += 5.0

In [ ]:
machine = build_machine_for_category(category_key="armour")
current_height += 5.0

In [ ]:
machine = build_machine_for_category(category_key="flight")
current_height += 5.0

In [ ]:
machine = build_machine_for_category(category_key="automation")
current_height += 5.0

In [ ]:
machine = build_machine_for_category(category_key="water")
current_height += 5.0

In [ ]:
machine = build_machine_for_category(category_key="space-flight")
current_height += 5.0

In [ ]:
machine.connect_blocks(
    block_a="1", 
    block_b="3", 
    face_a="f0", 
    face_b="f7", 
    connector="Brace"
)
machine.connect_blocks(
    block_a="1", 
    block_b="3", 
    face_a="f0", 
    face_b="f2", 
    connector="Spring"
)
machine.connect_blocks(
    block_a="1", 
    block_b="4", 
    face_a="f0", 
    face_b="f3", 
    connector="Rope Winch"
)
machine.connect_blocks(
    block_a="1", 
    block_b="4", 
    face_a="f0", 
    face_b="f8", 
    connector="Rope Measure"
)
machine.connect_blocks(
    block_a="1", 
    block_b="4", 
    face_a="f0", 
    face_b="f9", 
    connector="Fuel Line"
)
machine.to_file(output_dir=SAVED_MACHINE_DIR)

In [ ]:
# For each existing block, attach a Single Wooden Block to every attachable face.
# Freeze the current block IDs first so newly attached blocks are not re-iterated.
blocks_dict = machine.blocks() if callable(getattr(machine, "blocks", None)) else machine.blocks
if not isinstance(blocks_dict, dict):
    raise TypeError("machine.blocks is not a dict-like container")

base_block_ids = list(blocks_dict.keys())
attach_count = 0

for base_local_id in base_block_ids:
    base_block = blocks_dict.get(base_local_id)
    if base_block.name == "Starting Block":
        continue
    if base_block is None:
        raise ValueError(f"Block {base_local_id} not found during face traversal")

    attachable_faces = [
        face_name
        for face_name, face_obj in base_block.faces.items()
        if bool(face_obj.sticky)
    ]

    for face_name in attachable_faces:
        msg = machine.attach_block_to(
            base_block=base_local_id,
            face=face_name,
            new_block="Single Wooden Block",
        )
        print(msg)
        attach_count += 1

print(f"Attached Single Wooden Block on {attach_count} attachable faces across {len(base_block_ids)} base blocks.")
machine.name = "preview_all_att"
machine.to_file(output_dir=SAVED_MACHINE_DIR)